<a href="https://colab.research.google.com/github/AmirMohammad73/semantic_folding/blob/main/Umap_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install scipy

In [2]:
!pip install umap-learn
!pip install Levenshtein

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.4/177.4 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 14.5 MB/s eta 0:00:00


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [6]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import umap
import json
from nltk.corpus import wordnet

def standardize_synonyms(text):
    words = text.split()
    standardized_words = []
    for word in words:
        synsets = wordnet.synsets(word)
        if synsets:
            standardized_word = synsets[0].lemmas()[0].name()
            standardized_words.append(standardized_word)
        else:
            standardized_words.append(word)
    return " ".join(standardized_words)

# Load corpus
with open("/content/drive/MyDrive/cleaned_corpus_final1.txt", "r") as file:
    corpus = file.readlines()

# Standardize synonyms in the corpus
standardized_corpus = [standardize_synonyms(context) for context in corpus]

# Create a TF-IDF vectorizer
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(standardized_corpus)

# Get the vocabulary and its indices
vocabulary = vectorizer.get_feature_names_out()

# Initialize the document vectors
num_contexts = len(standardized_corpus)
num_vocab = len(vocabulary)
document_vectors = np.zeros((num_contexts, num_vocab))

# Fill in the document vectors with TF-IDF values
for i, context in enumerate(standardized_corpus):
    context_tfidf = tfidf_matrix[i].toarray().flatten()
    document_vectors[i] = context_tfidf

# # TF-IDF vectorization
# tfidf_vectorizer = TfidfVectorizer(max_features=4130)
# tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

# # Dimensionality reduction
# n_components = 4130
# svd = TruncatedSVD(n_components=n_components)
# document_vectors = svd.fit_transform(tfidf_matrix)


# Clustering with UMAP
umap_model = umap.UMAP(n_components=2)  # Set a fixed random seed
document_vectors_umap = umap_model.fit_transform(document_vectors)
# Step 1: Get the range for the document_vectors_umap
x_min, x_max = document_vectors_umap[:, 0].min(), document_vectors_umap[:, 0].max()
y_min, y_max = document_vectors_umap[:, 1].min(), document_vectors_umap[:, 1].max()

# Step 2: Create a 16x16 grid to bin the data
num_bins = 20
x_edges = np.linspace(x_min, x_max, num_bins)
y_edges = np.linspace(y_min, y_max, num_bins)

# Step 3: Assign document ID to the respective bins (the result will be indices of the bins)
x_bin_indices = np.digitize(document_vectors_umap[:, 0], bins=x_edges) - 1 # -1 to convert to 0-indexed
y_bin_indices = np.digitize(document_vectors_umap[:, 1], bins=y_edges) - 1 # -1 to convert to 0-indexed

# Step 4: Make a list of 16x16 grid coordinates plus document IDs
mapped_data = list(zip(x_bin_indices, y_bin_indices, range(len(document_vectors_umap))))

# Convert to a numpy array if necessary
mapped_data_array = np.array(mapped_data)

# Initialize a fourth column in 'mapped_data_array' for the frequency counts, setting them initially to zero
mapped_data_array = np.hstack((mapped_data_array, np.zeros((mapped_data_array.shape[0], 1), dtype=int)))


In [7]:
import math
# Define a function to calculate the Euclidean distance between two matrices
def calculate_euclidean_distance(matrix1, matrix2):
    # Ensure that both matrices have the same shape
    if matrix1.shape != matrix2.shape:
        raise ValueError("Matrices must have the same dimensions to calculate Euclidean distance.")

    # Flatten the matrices to 1D arrays
    matrix1_flat = matrix1.flatten()
    matrix2_flat = matrix2.flatten()

    # Calculate the Euclidean distance
    euclidean_distance = math.dist(matrix1_flat, matrix2_flat)
    return euclidean_distance

# Assuming 'matrices' is a list of 5 matrices where the first matrix is the question
# and the next four are the options
def compare_question_to_options_euclidean(matrices):
    question_matrix = matrices[0]
    options_matrices = matrices[1:]
    euclidean_distances = []

    # Calculate the Euclidean distance of the question to each option
    for option_matrix in options_matrices:
        distance = calculate_euclidean_distance(question_matrix, option_matrix)
        euclidean_distances.append(distance)

    # Since we want similarity, not distance, we can invert the distances
    # The smaller the distance, the higher the similarity
    euclidean_similarities = [1 / distance if distance != 0 else float('inf') for distance in euclidean_distances]

    return euclidean_similarities

In [8]:
from scipy.spatial import distance
# Define a function to calculate the Hamming distance between two matrices
def calculate_hamming_distance(matrix1, matrix2):
    # Ensure that both matrices have the same shape
    if matrix1.shape != matrix2.shape:
        raise ValueError("Matrices must have the same dimensions to calculate Hamming distance.")

    # Flatten the matrices to 1D arrays
    matrix1_flat = matrix1.flatten()
    matrix2_flat = matrix2.flatten()

    # Calculate the Hamming distance
    hamming_distance = distance.hamming(matrix1_flat, matrix2_flat)
    return hamming_distance

# Define a function to calculate the similarity based on Hamming distance
def calculate_hamming_similarity(matrix1, matrix2):
    hamming_distance = calculate_hamming_distance(matrix1, matrix2)
    hamming_similarity = 1 - (hamming_distance)
    return hamming_similarity

# Assuming 'matrices' is a list of 5 matrices where the first matrix is the question
# and the next four are the options
def compare_question_to_options_hamming(matrices):
    question_matrix = matrices[0]
    options_matrices = matrices[1:]
    hamming_similarities = []

    # Calculate the Hamming similarity of the question to each option
    for option_matrix in options_matrices:
        similarity = calculate_hamming_similarity(question_matrix, option_matrix)
        hamming_similarities.append(similarity)

    return hamming_similarities

In [9]:
# Define a function to calculate the Jaccard similarity between two matrices
def calculate_jaccard_similarity(matrix1, matrix2):
    # Ensure that both matrices have the same shape
    if matrix1.shape != matrix2.shape:
        raise ValueError("Matrices must have the same dimensions to calculate Jaccard similarity.")

    # Flatten the matrices to 1D arrays
    matrix1_flat = matrix1.flatten()
    matrix2_flat = matrix2.flatten()

    # Calculate the intersection and union of the matrices
    intersection = np.sum(np.minimum(matrix1_flat, matrix2_flat))
    union = np.sum(np.maximum(matrix1_flat, matrix2_flat))

    # Calculate the Jaccard similarity
    jaccard_similarity = intersection / union if union != 0 else 0
    return jaccard_similarity

# Assuming 'matrices' is a list of 5 matrices where the first matrix is the question
# and the next four are the options
def compare_question_to_options_jaccard(matrices):
    question_matrix = matrices[0]
    options_matrices = matrices[1:]
    jaccard_similarities = []

    # Calculate the Jaccard similarity of the question to each option
    for option_matrix in options_matrices:
        similarity = calculate_jaccard_similarity(question_matrix, option_matrix)
        jaccard_similarities.append(similarity)

    return jaccard_similarities

In [10]:
from numpy.linalg import norm
# Define a function to calculate the Cosine similarity between two matrices
def calculate_cosine_similarity(matrix1, matrix2):
    # Ensure that both matrices have the same shape
    if matrix1.shape != matrix2.shape:
        raise ValueError("Matrices must have the same dimensions to calculate Cosine similarity.")

    # Flatten the matrices to 1D arrays
    matrix1_flat = matrix1.flatten()
    matrix2_flat = matrix2.flatten()

    # Calculate the dot product and magnitudes of the matrices
    dot_product = np.dot(matrix1_flat, matrix2_flat)
    magnitude1 = norm(matrix1_flat)
    magnitude2 = norm(matrix2_flat)

    # Calculate the Cosine similarity
    cosine_similarity = dot_product / (magnitude1 * magnitude2) if magnitude1 != 0 and magnitude2 != 0 else 0
    return cosine_similarity

# Assuming 'matrices' is a list of 5 matrices where the first matrix is the question
# and the next four are the options
def compare_question_to_options_cosine(matrices):
    question_matrix = matrices[0]
    options_matrices = matrices[1:]
    cosine_similarities = []

    # Calculate the Cosine similarity of the question to each option
    for option_matrix in options_matrices:
        similarity = calculate_cosine_similarity(question_matrix, option_matrix)
        cosine_similarities.append(similarity)

    return cosine_similarities

In [11]:
from Levenshtein import distance as levenshtein_distance

# Define a function to calculate the Levenshtein distance between two matrices
def calculate_levenshtein_distance(matrix1, matrix2):
    # Ensure that both matrices have the same shape
    if matrix1.shape != matrix2.shape:
        raise ValueError("Matrices must have the same dimensions to calculate Levenshtein distance.")

    # Flatten the matrices to 1D arrays
    matrix1_flat = matrix1.flatten()
    matrix2_flat = matrix2.flatten()

    # Convert the arrays to strings
    matrix1_str = ''.join(map(str, matrix1_flat))
    matrix2_str = ''.join(map(str, matrix2_flat))

    # Calculate the Levenshtein distance
    levenshtein_distance_value = levenshtein_distance(matrix1_str, matrix2_str)

    return levenshtein_distance_value

# Define a function to calculate the similarity based on Levenshtein distance
def calculate_levenshtein_similarity(matrix1, matrix2):
    levenshtein_distance_value = calculate_levenshtein_distance(matrix1, matrix2)
    max_distance = max(matrix1.size, matrix2.size)
    levenshtein_similarity = 1 - (levenshtein_distance_value / max_distance)
    return levenshtein_similarity

# Assuming 'matrices' is a list of 5 matrices where the first matrix is the question
# and the next four are the options
def compare_question_to_options_levenshtein(matrices):
    question_matrix = matrices[0]
    options_matrices = matrices[1:]
    levenshtein_similarities = []

    # Calculate the Levenshtein similarity of the question to each option
    for option_matrix in options_matrices:
        similarity = calculate_levenshtein_similarity(question_matrix, option_matrix)
        levenshtein_similarities.append(similarity)

    return levenshtein_similarities

In [12]:
def calculate_sorensen_dice_index(matrix1, matrix2):
    # Ensure that both matrices have the same shape
    if matrix1.shape != matrix2.shape:
        raise ValueError("Matrices must have the same dimensions to calculate Sørensen-Dice index.")

    # Flatten the matrices to 1D arrays
    matrix1_flat = matrix1.flatten()
    matrix2_flat = matrix2.flatten()

    # Calculate the Sørensen-Dice index
    intersection = np.sum(np.minimum(matrix1_flat, matrix2_flat)) * 2
    total = np.sum(matrix1_flat) + np.sum(matrix2_flat)
    sorensen_dice_index = intersection / total if total != 0 else 0
    return sorensen_dice_index

# Assuming 'matrices' is a list of 5 matrices where the first matrix is the question
# and the next four are the options
def compare_question_to_options_sorensen_dice(matrices):
    question_matrix = matrices[0]
    options_matrices = matrices[1:]
    sorensen_dice_indices = []

    # Calculate the Sørensen-Dice index of the question to each option
    for option_matrix in options_matrices:
        index = calculate_sorensen_dice_index(question_matrix, option_matrix)
        sorensen_dice_indices.append(index)

    return sorensen_dice_indices

In [13]:
def matrix_to_sdr(matrix, num_largest):
    # Flatten the matrix into a 1D array
    flat_matrix = np.array(matrix).flatten()
    sorted_values = np.array(sorted(flat_matrix, reverse=True))
    temp_vector = flat_matrix
    result = np.zeros_like(flat_matrix)
    result_matrix = result.reshape(np.shape(matrix))  # Initialize result_matrix
    for i in range(num_largest):
        if sorted_values[i] == 0:
            break
        result[np.where(temp_vector == sorted_values[i])[0][0]] = 1
        temp_vector[np.where(temp_vector == sorted_values[i])[0][0]] = -1
        result_matrix = result.reshape(np.shape(matrix))
    return result_matrix


In [ ]:
# prompt: Write a function that returns the index of the largest value of an array as letters. That is, if the first element has the largest value, it returns the letter A. If the second element has the highest value, B and so on.

def largest_value_index_to_letter(array):
    # Find the index of the maximum value
    max_index = np.argmax(array)

    # Convert the index to a letter
    # A = 0, B = 1, C = 2, ...
    letter = chr(ord('A') + max_index)

    return letter

In [16]:

# Prompt the user for a question and its four options
inputs = ['water', 'liquid', 'food', 'fruit', 'hot']
# for i in range(5):
#     user_input = input("Enter question or option {}: ".format(i + 1))
#     inputs.append(user_input)

# Initialize a list to store the matrices for each input
matrices = []

# Initialize a frequency counter for each document with a dict comprehension
word_frequencies_list = []
for user_input in inputs:
    word_frequencies = {doc_id: 0 for doc_id in range(len(corpus))}
    # Scan through the corpus to find the exact frequency of each word in each document
    for doc_id, document in enumerate(corpus):
        # Convert the document to lowercase and split into words
        words = document.lower().split()
        # Count the occurrences of each word in the words list
        for word in user_input.split():
            word_frequencies[doc_id] += words.count(word.lower())
    word_frequencies_list.append(word_frequencies)

# Update 'mapped_data_array' to include the total word count for each input
for word_frequencies in word_frequencies_list:
    for i, (x_index, y_index, doc_id) in enumerate(mapped_data):
        total_word_count = word_frequencies[doc_id]
        mapped_data_array[i, 3] = total_word_count

    # Optionally, filter out the entries with a total word count of zero
    filtered_mapped_data = mapped_data_array[mapped_data_array[:, 3] > 0]

    # Initialize the 24x24 matrix with zeros
    matrix = np.zeros((num_bins, num_bins), dtype=int)

    # Iterate over the filtered_mapped_data to populate the matrix
    for data in filtered_mapped_data:
        x_coord, y_coord, doc_id, total_word_count = data
        matrix[y_coord, x_coord] += total_word_count  # Note that y_coord is used for rows, x_coord for columns
    # matrix = matrix_to_sdr(matrix, math.floor(0.02*num_bins**2))
    # print(matrix)
    # Append the matrix to the list of matrices
    matrices.append(matrix)
# Calculate similarities and print the result
euclidean_similarities = compare_question_to_options_euclidean(matrices)
hamming_similarities = compare_question_to_options_hamming(matrices)
jaccard_similarities = compare_question_to_options_jaccard(matrices)
cosine_similarities = compare_question_to_options_cosine(matrices)
levenshtein_distances = compare_question_to_options_levenshtein(matrices)
sorensen_dice_indices = compare_question_to_options_sorensen_dice(matrices)

print(f"euclidean similarity: {euclidean_similarities}")
print(f"hamming similarity: {hamming_similarities}")
print(f"jaccard similarity: {jaccard_similarities}")
print(f"cosine similarity: {cosine_similarities}")
print(f"leveneshtein similarity: {levenshtein_distances}")
print(f"sorensen similarity: {sorensen_dice_indices}")

# Write the matrices to a file
with open("/content/drive/MyDrive/UMAP_output.txt", "w") as f:
    for i, matrix in enumerate(matrices):
        # Write the input number
        f.write("Matrix for input {}:\n".format(i + 1))
        # Write each element of the matrix individually
        for row in matrix:
            for element in row:
                f.write("{:4d}".format(element))  # Adjust the format to your preference
            f.write("\n")
        f.write("\n")

euclidean similarity: [0.007818945268837432, 0.006190793459223146, 0.0068924551679557765, 0.007252663349700522]
hamming similarity: [0.6775, 0.6174999999999999, 0.6675, 0.6375]
jaccard similarity: [0.16416040100250626, 0.10430622009569378, 0.01, 0.10939357907253269]
cosine similarity: [0.46772183060523803, 0.11147416316763568, 0.012004039719474804, 0.3143171970546538]
leveneshtein similarity: [0.6799999999999999, 0.65, 0.65, 0.6575]
sorensen similarity: [0.28202368137782563, 0.18890814558058924, 0.019801980198019802, 0.19721329046087888]
